# GEE ERA5-Land Downloader - Multi-Variable Hourly (Per-Variable Stacking)
Notebook ini didesain khusus untuk mengunduh data reanalisis atmosfer per-jam **ERA5-Land Hourly** (`ECMWF/ERA5_LAND/HOURLY`) dari Google Earth Engine (GEE) untuk 6 variabel atmosfer utama:
1. **`precipitation`**: Curah hujan (m -> **mm**, `x1000`)
2. **`temperature_2m`**: Suhu udara 2m (K -> **°C**, `-273.15`)
3. **`dewpoint_temperature_2m`**: Suhu titik embun 2m (K -> **°C**, `-273.15`)
4. **`u_wind_10m`**: Zonal wind / angin U 10m (**m/s**)
5. **`v_wind_10m`**: Meridional wind / angin V 10m (**m/s**)
6. **`surface_pressure`**: Tekanan permukaan (Pa -> **hPa**, `/100`)

### Fitur Utama:
- **Dual Compatibility:** Dapat berjalan di environment **Lokal** maupun **Kaggle Notebook** secara otomatis.
- **GeoJSON Boundary:** Memotong data sesuai poligon batas wilayah (`33.05_kecamatan.geojson`).
- **Smart Cache / Input Check:** Mendeteksi file yang sudah diunduh sebelumnya di Kaggle Input Datasets.
- **Per-Variable Stacking:** Mengunduh 6 variabel secara individual (744 band/var $\le$ 1024 limit GEE) lalu digabungkan menjadi 1 file NetCDF multi-variabel.
- **Struktur Hierarkis 4-Tier:** Menyiapkan folder `data/era5_land/<tahun>/` secara sistematis.

In [ ]:
!pip install geemap earthengine-api rioxarray geopandas xarray netCDF4 requests --quiet

In [ ]:
import ee
import geopandas as gpd
import geemap
from shapely.validation import make_valid
from shapely.ops import transform, unary_union
import os
import shutil
import glob
import rioxarray as rxr
import xarray as xr
import pandas as pd
import calendar
from datetime import datetime, timedelta

# ==========================================
# 0. PENGATURAN DIREKTORI DINAMIS & PARAMETER
# ==========================================
BASE_DIR = os.getcwd()

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    file_geojson = "/kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson"
    if not os.path.exists(file_geojson):
        found_geo = glob.glob("/kaggle/input/**/33.05_kecamatan.geojson", recursive=True)
        file_geojson = found_geo[0] if found_geo else os.path.join(BASE_DIR, "33.05_kecamatan.geojson")
else:
    file_geojson = os.path.join(BASE_DIR, "33.05_kecamatan.geojson")

FOLDER_BASE_OUTPUT = os.path.join(BASE_DIR, "data", "era5_land")

# Direktori Dataset ERA5-Land yang sudah diunduh sebelumnya di Kaggle (Smart Cache)
KAGGLE_ERA5_INPUT = "/kaggle/input/datasets/jerismeteo/gee-era5-land-kebumen/data/era5_land"

tahun_awal = 2000
tahun_akhir = 2026

print(f"📌 File GeoJSON        : {file_geojson}")
print(f"📌 Folder Base Output  : {FOLDER_BASE_OUTPUT}")
print(f"📌 Folder Input Kaggle : {KAGGLE_ERA5_INPUT}")
print(f"📌 Periode Pengunduhan : {tahun_awal} s.d. {tahun_akhir}")


In [ ]:
# =========================================================================
# 1. INISIALISASI GOOGLE EARTH ENGINE (GEE)
# Mendukung: 1. Service Account Lokal (.json) | 2. Kaggle Secrets (GEE_KEY) | 3. Fallback Auth
# =========================================================================
import os
import json
import ee
from google.oauth2.service_account import Credentials

def init_earth_engine(project_id='staklimjerukagung'):
    # 1. Coba via Service Account File Lokal (Repo Projek_Downscale)
    sa_candidates = [
        os.path.join(os.getcwd(), "staklimjerukagung-b852a12a367e.json"),
        os.path.join(os.path.dirname(os.getcwd()), "staklimjerukagung-b852a12a367e.json"),
        "staklimjerukagung-b852a12a367e.json"
    ]
    for sa_path in sa_candidates:
        if os.path.exists(sa_path):
            try:
                with open(sa_path, 'r') as f:
                    sa_info = json.load(f)
                SCOPES = ['https://www.googleapis.com/auth/earthengine']
                credentials = Credentials.from_service_account_info(sa_info, scopes=SCOPES)
                ee.Initialize(credentials=credentials, project=project_id)
                print(f"✅ Berhasil Inisialisasi GEE via Local SA File: {os.path.basename(sa_path)}")
                return
            except Exception as e:
                print(f"⚠️ Gagal via {sa_path}: {e}")

    # 2. Coba via Kaggle Secrets (Jika di-run di Kaggle Kernel)
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        sa_info = json.loads(user_secrets.get_secret("GEE_KEY"))
        SCOPES = ['https://www.googleapis.com/auth/earthengine']
        credentials = Credentials.from_service_account_info(sa_info, scopes=SCOPES)
        ee.Initialize(credentials=credentials, project=project_id)
        print("✅ Berhasil Inisialisasi GEE via Kaggle Secret (GEE_KEY)")
        return
    except Exception:
        pass

    # 3. Fallback ke autentikasi default / browser interaktif
    try:
        ee.Initialize(project=project_id)
        print("✅ Berhasil Inisialisasi GEE via Default Session")
    except Exception as e:
        print(f"⚠️ Inisialisasi default gagal ({e}), membuka autentikasi browser...")
        ee.Authenticate()
        ee.Initialize(project=project_id)

init_earth_engine()


In [ ]:
# ==========================================
# 2. PERSIAPAN BATAS WILAYAH (GEOJSON CLEAN)
# ==========================================
if not os.path.exists(file_geojson):
    raise FileNotFoundError(f"File GeoJSON tidak ditemukan di: {file_geojson}")

gdf = gpd.read_file(file_geojson)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

print("Membersihkan geometri GeoJSON yang cacat...")
gdf = gdf[gdf.geometry.notna()].copy()

def _to_2d(geom):
    if geom is None or geom.is_empty: return None
    return transform(lambda x, y, z=None: (x, y), geom)

def _clean_geom(geom):
    if geom is None or geom.is_empty: return None
    geom = _to_2d(geom)
    geom = make_valid(geom)
    if geom is None or geom.is_empty: return None
    return geom.buffer(0)

gdf["geometry"] = gdf["geometry"].apply(_clean_geom)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf.reset_index(drop=True, inplace=True)

print("Mengonversi ke Earth Engine...")
geojson_fc = gdf.__geo_interface__
batas_kebumen = ee.FeatureCollection(geojson_fc["features"])
bounds = gdf.total_bounds
ee_bbox = ee.Geometry.BBox(bounds[0] - 0.01, bounds[1] - 0.01, bounds[2] + 0.01, bounds[3] + 0.01)
print(f"✅ Batas Wilayah Siap! Extent: Lon [{bounds[0]:.4f}, {bounds[2]:.4f}], Lat [{bounds[1]:.4f}, {bounds[3]:.4f}]")

In [ ]:
# ==========================================
# 3. FUNGSI UNDUH SUPER-CEPAT ERA5-LAND (PER-VARIABLE STACKING)
# ==========================================
def cari_file_input_era5(tahun, bulan):
    """Mengecek apakah file NetCDF ERA5-Land sudah tersedia di Input Dataset Kaggle."""
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        possible_paths = [
            f"{KAGGLE_ERA5_INPUT}/{tahun}/era5_land_{tahun}_{bulan:02d}.nc",
            f"{KAGGLE_ERA5_INPUT}/{tahun}/{tahun}_{bulan:02d}/era5_land_{tahun}_{bulan:02d}.nc",
            f"/kaggle/input/datasets/jerismeteo/gee-era5-land-kebumen/{tahun}/era5_land_{tahun}_{bulan:02d}.nc",
            f"/kaggle/input/datasets/jerismeteo/gee-era5-land-kebumen/era5_land_{tahun}_{bulan:02d}.nc"
        ]
        for p in possible_paths:
            if os.path.exists(p):
                return p
    return None

def unduh_era5_land_bulanan(tahun, bulan, batas_ee, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    nc_path = os.path.join(output_dir, f"era5_land_{tahun}_{bulan:02d}.nc")
    
    # 1. Cek di working directory
    if os.path.exists(nc_path):
        print(f"[{tahun}-{bulan:02d}] File sudah ada di working directory, dilewati...")
        return
        
    # 2. Cek di Kaggle Input Datasets (Smart Cache)
    input_file = cari_file_input_era5(tahun, bulan)
    if input_file and os.path.exists(input_file):
        print(f"[{tahun}-{bulan:02d}] ✓ Ditemukan di Input Kaggle: {input_file}")
        print(f"[{tahun}-{bulan:02d}] Menyalin file ke {nc_path}...")
        shutil.copy2(input_file, nc_path)
        return
        
    tgl_mulai = f"{tahun}-{bulan:02d}-01"
    if bulan == 12:
        tgl_akhir = f"{tahun+1}-01-01"
    else:
        tgl_akhir = f"{tahun}-{bulan+1:02d}-01"
        
    print(f"[{tahun}-{bulan:02d}] Menarik data ImageCollection ERA5-Land dari GEE...")
    try:
        col = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
               .filterBounds(ee_bbox)
               .filterDate(tgl_mulai, tgl_akhir))
        
        if col.size().getInfo() == 0:
            print(f"[{tahun}-{bulan:02d}] ⚠️ Data tidak tersedia di GEE untuk periode ini.")
            return
            
        timestamps = col.aggregate_array("system:time_start").getInfo()
        dates = pd.to_datetime(timestamps, unit='ms')
        
        # Konfigurasi 6 Variabel & Transformasi Satuan
        vars_config = [
            ('precipitation', 'total_precipitation_hourly', lambda img: img.multiply(1000)),
            ('temperature_2m', 'temperature_2m', lambda img: img.subtract(273.15)),
            ('dewpoint_temperature_2m', 'dewpoint_temperature_2m', lambda img: img.subtract(273.15)),
            ('u_wind_10m', 'u_component_of_wind_10m', lambda img: img),
            ('v_wind_10m', 'v_component_of_wind_10m', lambda img: img),
            ('surface_pressure', 'surface_pressure', lambda img: img.divide(100))
        ]
        
        ds_out = xr.Dataset()
        temp_tifs = []
        
        for var_out_name, gee_band_name, transform_fn in vars_config:
            tif_var_temp = os.path.join(output_dir, f"temp_era5_{var_out_name}_{tahun}_{bulan:02d}.tif")
            temp_tifs.append(tif_var_temp)
            
            col_var = col.select(gee_band_name).map(lambda img: transform_fn(img).rename(var_out_name))
            stacked_var_img = col_var.toBands().clip(ee_bbox)
            
            # Export per variabel (max 744 band <= 1024)
            geemap.ee_export_image(
                stacked_var_img,
                filename=tif_var_temp,
                region=ee_bbox,
                scale=11132,
                file_per_band=False
            )
            
            with rxr.open_rasterio(tif_var_temp, masked=True) as da_var:
                da_var = da_var.rename({'band': 'time'})
                da_var['time'] = dates[:len(da_var.time)]
                da_var.name = var_out_name
                ds_out[var_out_name] = da_var.load()
                
        # Simpan ke NetCDF multi-variabel utuh
        ds_out.to_netcdf(nc_path)
        ds_out.close()
        
        # Hapus file sementara
        for temp_tif in temp_tifs:
            if os.path.exists(temp_tif):
                os.remove(temp_tif)
                
        print(f"[{tahun}-{bulan:02d}] ✓ Selesai tersimpan (6 variabel): {nc_path}\n")
    except Exception as e:
        print(f"[{tahun}-{bulan:02d}] ❌ Error: {e}")
        for var_out_name, _, _ in vars_config:
            tif_var_temp = os.path.join(output_dir, f"temp_era5_{var_out_name}_{tahun}_{bulan:02d}.tif")
            if os.path.exists(tif_var_temp):
                os.remove(tif_var_temp)

def unduh_era5_multi_tahun(tahun_awal, tahun_akhir, batas_ee, output_base_dir):
    print(f"\n{'='*60}")
    print(f"UNDUH ERA5-LAND HOURLY MULTI-VARIABEL: {tahun_awal} - {tahun_akhir}")
    print(f"{'='*60}\n")
    
    for tahun in range(tahun_awal, tahun_akhir + 1):
        folder_tahun = os.path.join(output_base_dir, str(tahun))
        for bulan in range(1, 13):
            unduh_era5_land_bulanan(tahun, bulan, batas_ee, folder_tahun)
            
    print(f"{'='*60}")
    print("🎉 SELESAI UNDUH SELURUH TAHUN ERA5-LAND")
    print(f"{'='*60}")


In [ ]:
# ==========================================
# 4. EKSEKUSI FUNGSI UTAMA ERA5-LAND
# ==========================================
unduh_era5_multi_tahun(
    tahun_awal=tahun_awal,
    tahun_akhir=tahun_akhir,
    batas_ee=batas_kebumen,
    output_base_dir=FOLDER_BASE_OUTPUT
)
